In [1]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset

Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 157M/157M [00:01<00:00, 159MB/s]



In [2]:
import zipfile
zip_ref = zipfile.ZipFile('/content/brain-tumor-mri-dataset.zip', 'r')
zip_ref.extractall("/content")
zip_ref.close()

In [3]:
import tensorflow as tf
from tensorflow import keras
from keras.layers import Input, Dense, Dropout, Flatten, RandomFlip, RandomContrast, RandomRotation, RandomZoom, Rescaling
from keras.optimizers import Adam
from keras.applications import VGG16
from keras.models import Sequential
from keras.preprocessing.image import load_img, img_to_array
from sklearn.utils import shuffle
from tensorflow.keras.applications.vgg16 import preprocess_input

In [4]:
train_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/Training',
    labels = 'inferred',
    label_mode = 'int',
    batch_size = 32,
    image_size = (256,256)
)

test_ds = keras.utils.image_dataset_from_directory(
    directory = '/content/Testing',
    labels = 'inferred',
    label_mode = 'int',
    batch_size = 32,
    image_size = (256,256)
)

Found 5600 files belonging to 4 classes.
Found 1600 files belonging to 4 classes.


In [5]:
train_images = []
train_labels = []

for batch_images, batch_labels in train_ds:
    for img, label in zip(batch_images, batch_labels):
        train_images.append(img)
        train_labels.append(label)

print(len(train_images))
print(len(train_labels))

train_images, train_labels = shuffle(train_images, train_labels)

5600
5600


In [6]:
test_images = []
test_labels = []

for batch_images, batch_labels in test_ds:
    for img, label in zip(batch_images, batch_labels):
        test_images.append(img)
        test_labels.append(label)

print(len(test_images))
print(len(test_labels))

test_images, test_labels = shuffle(test_images, test_labels)

1600
1600


In [7]:
IMAGE_SIZE = 256

data_augmentation = Sequential([
    RandomFlip('horizontal'),
    RandomRotation(0.1),
    RandomZoom(0.1),
    RandomContrast(0.2)
])

train_ds = train_ds.map(
    lambda x, y: (preprocess_input(x), y)
)

test_ds = test_ds.map(
    lambda x, y: (preprocess_input(x), y)
)


base_model = VGG16(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False, weights='imagenet')

base_model.trainable = False

base_model.layers[-2].trainable = True
base_model.layers[-3].trainable = True
base_model.layers[-4].trainable = True

model = Sequential([
    Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),

    data_augmentation,

    base_model,

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.1),

    Dense(64, activation="relu"),
    Dropout(0.1),

    Dense(4, activation="softmax")
])


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [8]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 8, 8, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,917,636 (72.17 MB)

 Trainable params: 11,282,372 (43.04 MB)

 Non-trainable params: 7,635,264 (29.13 MB)

In [11]:
model.compile(optimizer=Adam(learning_rate=0.0001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(train_ds, epochs = 5, validation_data=test_ds)

Epoch 1/5
  2/175 ━━━━━━━━━━━━━━━━━━━━ 1:27:21 30s/step - accuracy: 0.3281 - loss: 3.2942

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], color='red', label='train')
plt.plot(history,history['val_accuracy'], color='blue', label='validation')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['loss'], color='red', label='train')
plt.plot(history,history['val_loss'], color='blue', label='validation')
plt.legend()
plt.show()

In [37]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from keras.models import load_model
import numpy as np

In [35]:
test_predictions = model.predict(test_images)
print('Classification Report:')
print(classification_report(test_labels, np.argmax(test_predictions, axis=1)))

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: keras_tensor_97
Received: inputs=('Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256, 3))', 'Tensor(shape=(32, 256,

ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("data:0", shape=(32, 256, 3), dtype=float32) with name 'keras_tensor_97' and path ''. Expected shape (None, 256, 256, 3), but input has incompatible shape (32, 256, 3)[0m

Arguments received by Sequential.call():
  • inputs=('tf.Tensor(shape=(32, 256, 3), dtype=float32)', 'tf.Tensor(shape=(32, 256, 3), dtype=float32)', 'tf.Tensor(shape=(32, 256, 3), dtype=float32)', 'tf.Tensor(shape=(32, 256, 3), dtype=float32)', 'tf.Tensor(shape=(32, 256, 3), dtype=float32)', 'tf.Tensor(shape=(32, 256, 3), dtype=float32)', 'tf.Tensor(shape=(32, 256, 3), dtype=float32)', 'tf.Tensor(shape=(32, 256, 3), dtype=float32)', 'tf.Tensor(shape=(32, 256, 3), dtype=float32)', 'tf.Tensor(shape=(32, 256, 3), dtype=float32)', 'tf.Tensor(shape=(32, 256, 3), dtype=float32)', 'tf.Tensor(shape=(32, 256, 3), dtype=float32)', 'tf.Tensor(shape=(32, 256, 3), dtype=float32)', 'tf.Tensor(shape=(32, 256, 3), dtype=float

In [ ]:
conf_matrix = confusion_matrix(test_labels, np.argmax(test_predictions, axis=1))
print("Confusion Matrix:")
print(conf_matrix)

In [ ]:
import os
train_dir = '/content/Training'
test_dir = '/content/Testing'
plt.figure(figsize=(5,5))
sns.heatmap(conf_matrix, annot=True, cmap='Blues', xticklabels=os.listdir(train_dir), yticklabels=os.listdir(train_dir))
sns.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

In [ ]:
model.save('model.pkl')

In [ ]:
from tensorflow.keras.models import load_model
model = load_model('model.pkl')

In [ ]:
class_labels = ['glioma', 'meningioma', 'notumor', 'pituitary']

def predict(img_path, model, image_size=256):
  try:
    img = load_img(img_path, target_size=(image_size, image_size))
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array)
    predicted_class = np.argmax(prediction, axis=1)[0]
    confidence_score = np.max(prediction, axis=1)[0]

    if class_labels[predicted_class] == 'notumor':
      result = 'No Tumor'

    else:
      result = f"Tumor: {class_labels[predicted_class]}"

    plt.imshow(load_img(img_path))
    plt.axis('off')
    plt.title(f"{result} (Confidence: {confidence_score*100 : .2f}%)")
    plt.show()

  except Exception as e:
    print('Error in processing the image:', str(e))



In [ ]:
image_path = '/content/meningioma-tumor.jpg'
predict(image_path, model)